# 25. 새 질문 10개로 발췌 답변 확인하기

[프로젝트 적용] 기존 개인 DB·로컬 모델·M7 검색을 사용했습니다.
[프로젝트 추가] 실행 전에 정한 상태·출처·문구·그림 연결 조건을 실제 반환 결과와 비교했습니다. 기존 답변 규칙은 바꾸지 않았습니다.

**여기서는 완료된 결과 파일만 읽습니다.** DB 조회·모델 실행·재임베딩·그림 업로드를 하지 않습니다. 첫 코드 셀부터 차례대로 실행하세요.

실행 설정은 purpose_specific·상위 후보 5개입니다. 기본 발췌 명령의 후보 3개 설정과 구분합니다. 같은 주제의 새 표현을 포함한 확인 작업이며 독립적인 전체 성능 평가가 아닙니다. 그림의 반환 주소·페이지 연결을 확인했으며 그림 픽셀의 의미를 채점한 것은 아닙니다.


In [ ]:
from pathlib import Path
import json

# notebooks에서 실행해도 프로젝트의 결과 파일을 찾도록 경로를 맞춥니다.
project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
report_path = project / "data/zzong_santafe_lag/reports/answer_checks_10_20261002_144711.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
if report["phase"] != "completed" or report["completed_count"] != 10:
    raise ValueError("실행 오류 없이 10개 결과가 완료됐는지 먼저 확인하세요.")
print("확인 완료 질문 수:", report["completed_count"])
print("기대 조건에 맞은 질문:", report["expectation_match_count"])
print("보완할 질문:", report["expectation_mismatch_count"])
print("실행 오류 수:", report["operational_error_count"])


## 먼저 전체 결과 읽기

예상 출력은 **완료 10개 / 기대 조건 일치 7개 / 보완 3개 / 실행 오류 0개**입니다. 7개는 상태·출처·문구·그림 참조의 자동 조건에 맞았다는 뜻입니다. 답변 정확도 70%로 해석하지 않습니다.

상태 뜻: evidence_excerpt=확인 글 발췌, needs_review=원문 확인 필요, outside_pdf_scope=PDF로 판단할 수 없는 요청입니다.


In [ ]:
# 자동 기대 조건과 실제 상태·출처·그림 개수를 함께 읽습니다.
for row in report["results"]:
    print("\n", row["case_id"], row["question"])
    print("기대 상태:", row["expected"]["expected_status"], "| 실제:", row["actual_status"])
    print("출처 PDF 쪽:", row["actual_source_pages"])
    print("그림 표시 참조:", row["displayed_image_count"], "| 보류 참조:", row["pending_image_count"])
    print("기대 조건에 맞나요?:", row["expectation_checks_passed"])


## 보완할 세 문제

- **A04:** 오일 주입량과 점도 표의 온도 눈금을 함께 요청했지만 점도 표만 발췌했습니다. 43쪽 주입량 근거가 빠졌습니다.
- **A05:** 차대번호와 미러 버튼 위치를 함께 요청했지만 미러 안내도만 발췌했습니다. 45쪽 차대번호 근거가 빠졌습니다.
- **A06:** 현재 내 차량의 리콜 대상 여부를 물었는데 PDF 밖 요청 대신 검토 필요로 처리했습니다. 리콜 여부를 지어낸 것은 아니지만 이유를 명확히 안내하지 못했습니다.

아래는 한 문제의 실제 발췌를 읽는 셀입니다. 처음에는 A04만 확인하고, case_id를 A05 또는 A06으로 바꾸어 나머지도 읽을 수 있습니다.


In [ ]:
# 한 번에 한 문제의 실제 결과를 읽습니다. 질문을 다시 실행하는 셀이 아닙니다.
case_id = "A04"
selected = next(row for row in report["results"] if row["case_id"] == case_id)
print("질문:", selected["question"])
print("기대 출처 그룹:", selected["expected"].get("required_page_groups", []))
print("실제 출처:", selected["actual_source_pages"])
print("\n실제 답변:\n", selected["answer"]["answer"])
if selected["answer"].get("reason"):
    print("보류 이유:", selected["answer"]["reason"])


## 그림 요청의 결과

A09 차대번호 요청은 PDF 45쪽의 공개 그림 참조 1개를 반환했습니다. A10 실외 미러 버튼 요청은 PDF 33~34쪽 안내도를 찾았지만 그림 파일은 아직 업로드하지 않아 참조 1개를 보류했습니다. 그림이 없을 때 다른 그림을 대신 붙이지 않는지도 확인했습니다.


In [ ]:
# 공개 파일을 새로 내려받지 않고 반환된 그림 페이지·업로드 상태만 살펴봅니다.
for row in report["results"]:
    if row["case_id"] in {"A09", "A10"}:
        print("\n", row["case_id"], row["question"])
        for image in row["answer"]["images"]:
            print("표시 가능:", image["pdf_page_number"], image["file_name"], "| 주소 있음:", bool(image["public_url"]))
        for image in row["answer"]["pending_image_references"]:
            print("표시 보류:", image["pdf_page_number"], image["file_name"], "| 업로드 상태:", image["upload_status"])


## 다음 수정 검토안 — 아직 적용 전

1. 두 내용을 함께 묻는 요청은 필요한 근거를 나누어 찾고 각각의 출처를 표시합니다. 한쪽만 확보했으면 나머지는 확인 불가라고 분명히 안내합니다.
2. 현재 개별 차량의 리콜 대상 조회는 PDF 밖으로 분류하고 PDF 한 개로 확인할 수 없는 이유를 설명합니다. PDF 안의 리콜 안내 문구를 묻는 질문과 구분해야 합니다.
3. 그림 파일이 아직 없으면 업로드·연결 확인 전이라는 이유를 사용자에게 보여줍니다.

실행 코드는 answer_checks.py입니다. 이번 결과는 수정 전 기준으로 보관하며 답변 규칙 수정은 사용자 확인 후 진행합니다. 미검토 자료를 평가 점수 때문에 검토 완료로 바꾸지 않습니다.
